# 🔍 Local RAG — PDF → PostgreSQL (pgvector) → gpt-oss-20b via HuggingFace

**Stack:**
- 📄 PDF parsing → `pypdf`
- 🧠 Embeddings → Gemini `gemini-embedding-001` (free API)
- 🗄️ Vector DB → `PostgreSQL 18 + pgvector` (local, persistent)
- 💬 LLM → `gpt-oss-20b` via HuggingFace Inference Providers (Groq)
- 🧠 Memory → Conversation checkpointing to PostgreSQL tables
- 🔄 CDC → Chunk-level Change Data Capture (PostgreSQL-backed)
- ⏱️ Recency → Exponential decay re-ranking via SQL

---

#### ── Core Pipeline ──
- Cell 1  — Install dependencies
- Cell 2  — Config & API keys + PostgreSQL connection
- Cell 3  — PDF extraction
- Cell 4  — Chunking
- Cell 5  — Gemini batch embedding
- Cell 6  — PostgreSQL storage & retrieval (pgvector)
- Cell 7  — ▶️ Run ingestion (point to your PDF here)
- Cell 8  — gpt-oss-20b answer generation (HuggingFace)
- Cell 9  — Full RAG pipeline — `ask()`
- Cell 10 — ▶️ Ask a single question
- Cell 11 — ▶️ Interactive REPL loop
- Cell 12 — ▶️ Utilities (stats, list docs, wipe DB)

#### ── PostgreSQL Explorer ──
- Cell 13  — ▶️ Direct PostgreSQL query explorer

#### ── Conversation Memory ──
- Cell 13A — Memory Manager (`ConversationMemory` class — PostgreSQL backed)
- Cell 13B — Memory-aware `ask_with_memory()`
- Cell 13C — ▶️ Interactive memory REPL
- Cell 13D — ▶️ Initialize memory session (run once per session)
- Cell 13E — ▶️ Ask a single question with memory
- Cell 13F — ▶️ Ask multiple questions with memory (follow-ups)

#### ── Staleness · CDC · Recency ──
- Cell 14A — Staleness Registry + enhanced `store_in_postgres_v2()`
- Cell 14B — CDC Engine + smart `ingest_pdf_v2()` entry point
- Cell 14C — Recency-weighted retrieval + `ask_weighted()`
- Cell 14D — ▶️ Test PDF generator (creates v1 / v2 / v3)
- Cell 14E — ▶️ Full test suite (staleness · CDC · recency · memory)

---

**Feature summary:**

| Feature | Cell | Description |
|---|---|---|
| Standard RAG | 1–11 | Ingest PDF → embed → retrieve → answer |
| DB Explorer | 13 | SQL-based peek, filter, keyword search, export |
| Memory | 13A–13F | Conversation history in PostgreSQL tables |
| Staleness | 14A | SHA256 file hash in rag_staleness table |
| CDC | 14B | Only changed chunks re-embedded — rag_chunk_registry |
| Recency | 14C | Newer chunks ranked higher via exponential decay |
| Testing | 14D–14E | Multi-version PDFs + end-to-end test suite |

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run once. Restart kernel after installation completes.

# %pip install google-genai pypdf pgvector psycopg2-binary rich python-dotenv huggingface_hub fpdf2

In [1]:
# ── Cell 2: Config & API Keys + PostgreSQL Connection ─────────────────────────
#
# .env file should contain:
#   GEMINI_API_KEY=your_gemini_key_here
#   HF_API_KEY=your_huggingface_token_here
#   PG_PASSWORD=your_postgres_password

import os
from dotenv import load_dotenv
import psycopg2
from pgvector.psycopg2 import register_vector

load_dotenv()

# ── API Keys ──────────────────────────────────────────────────────────────────
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY", "")
HF_API_KEY         = os.environ.get("HF_API_KEY", "")
EMBED_DIM          = 768

# ── Model config ──────────────────────────────────────────────────────────────
GEMINI_EMBED_MODEL = "gemini-embedding-001"
HF_LLM_MODEL       = "openai/gpt-oss-20b:groq"

# ── PostgreSQL connection config ──────────────────────────────────────────────
# Update PG_PASSWORD below or add PG_PASSWORD=yourpassword to your .env file
PG_HOST     = "localhost"
PG_PORT     = 5432
PG_DB       = "postgres"
PG_USER     = "postgres"
PG_PASSWORD = os.environ.get("PG_PASSWORD", "postgres")  # change if needed

# ── Chunking config ───────────────────────────────────────────────────────────
CHUNK_SIZE         = 800
CHUNK_OVERLAP      = 120

# ── Retrieval config ──────────────────────────────────────────────────────────
TOP_K              = 5

# ── Embedding config ──────────────────────────────────────────────────────────
EMBED_BATCH_SIZE   = 50
BATCH_SLEEP_SEC    = 0.3

# ── LLM generation config ─────────────────────────────────────────────────────
LLM_MAX_NEW_TOKENS = 1024
LLM_TEMPERATURE    = 0.1

# ── Validate keys ─────────────────────────────────────────────────────────────
assert GEMINI_API_KEY, "❌ GEMINI_API_KEY not set."
assert HF_API_KEY,     "❌ HF_API_KEY not set."

# ── PostgreSQL connection ─────────────────────────────────────────────────────
def get_pg_conn():
    """Returns a fresh PostgreSQL connection with pgvector registered."""
    conn = psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        dbname=PG_DB, user=PG_USER, password=PG_PASSWORD
    )
    register_vector(conn)
    return conn

# Test connection
try:
    _test = get_pg_conn()
    cur   = _test.cursor()
    cur.execute("SELECT version();")
    pg_ver = cur.fetchone()[0].split(",")[0]
    cur.execute("SELECT COUNT(*) FROM rag_chunks;")
    chunk_count = cur.fetchone()[0]
    _test.close()
    print(f"✅ PostgreSQL connected: {pg_ver}")
    print(f"   rag_chunks rows : {chunk_count}")
except Exception as e:
    print(f"❌ PostgreSQL connection failed: {e}")
    print("   Check PG_PASSWORD and that PostgreSQL is running.")

print(f"\n✅ Config loaded")
print(f"   Embedding model : {GEMINI_EMBED_MODEL}")
print(f"   LLM model       : {HF_LLM_MODEL}")
print(f"   PostgreSQL      : {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   Chunk size      : {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")

✅ PostgreSQL connected: PostgreSQL 18.3 on x86_64-windows
   rag_chunks rows : 0

✅ Config loaded
   Embedding model : gemini-embedding-001
   LLM model       : openai/gpt-oss-20b:groq
   PostgreSQL      : postgres@localhost:5432/postgres
   Chunk size      : 800 chars | Overlap: 120 chars


In [2]:
# ── Cell 3: PDF Text Extraction ───────────────────────────────────────────────

from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> tuple[str, int]:
    """
    Extracts full text from a PDF file.
    Tags each page so chunks stay traceable back to their page number.
    Returns: (full_text, page_count)
    """
    reader = PdfReader(pdf_path)
    pages  = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append(f"[Page {i + 1}]\n{text.strip()}")
    return "\n\n".join(pages), len(reader.pages)

print("✅ extract_text_from_pdf() defined")

✅ extract_text_from_pdf() defined


In [3]:
# ── Cell 4: Text Chunking ─────────────────────────────────────────────────────

def chunk_text(text: str) -> list[str]:
    """
    Sliding window chunker with overlap.
    Drops fragments under 80 chars (stray headers, page numbers, etc.).
    """
    chunks, start = [], 0
    while start < len(text):
        end   = start + CHUNK_SIZE
        chunk = text[start:end].strip()
        if len(chunk) >= 80:
            chunks.append(chunk)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

print("✅ chunk_text() defined")

✅ chunk_text() defined


In [4]:
# ── Cell 5: Gemini Batch Embedding ────────────────────────────────────────────

import time
from google import genai
from google.genai import types

genai_client = genai.Client(api_key=GEMINI_API_KEY)


def embed_documents_batch(chunks: list[str]) -> list[list[float]]:
    """
    Embeds document chunks via Gemini API one at a time.
    task_type='RETRIEVAL_DOCUMENT' for stored passages.
    output_dimensionality=768 — MRL truncation from 3072.
    """
    all_embeddings = []
    total          = len(chunks)

    for i, chunk in enumerate(chunks, 1):
        print(f"  Embedding chunk {i}/{total}...", end="\r")
        try:
            result = genai_client.models.embed_content(
                model=GEMINI_EMBED_MODEL,
                contents=chunk,
                config=types.EmbedContentConfig(
                    task_type="RETRIEVAL_DOCUMENT",
                    output_dimensionality=EMBED_DIM,
                ),
            )
            all_embeddings.append(result.embeddings[0].values)
            time.sleep(0.05)
        except Exception as e:
            print(f"\n  ❌ Chunk {i} skipped: {e}")
            all_embeddings.append([0.0] * EMBED_DIM)

    print(f"\n  ✅ Embedded {len(all_embeddings)} chunks")
    return all_embeddings


def embed_query(text: str) -> list[float]:
    """
    Embeds a user query.
    task_type='RETRIEVAL_QUERY' — asymmetric pair with RETRIEVAL_DOCUMENT.
    """
    result = genai_client.models.embed_content(
        model=GEMINI_EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=EMBED_DIM,
        ),
    )
    return result.embeddings[0].values


print(f"✅ embed_documents_batch() and embed_query() defined")
print(f"   Model: {GEMINI_EMBED_MODEL} | Dimensions: {EMBED_DIM}")

✅ embed_documents_batch() and embed_query() defined
   Model: gemini-embedding-001 | Dimensions: 768


In [5]:
# ── Cell 6: PostgreSQL (pgvector) Storage & Retrieval ─────────────────────────
# Replaces ChromaDB entirely.
# Uses rag_chunks table with vector(768) column and HNSW cosine index.

import numpy as np


def store_in_postgres(
    chunks:      list[str],
    embeddings:  list[list[float]],
    doc_name:    str,
) -> int:
    """
    Inserts chunk text + embedding vectors into rag_chunks.
    Returns the number of rows inserted.
    """
    conn = get_pg_conn()
    cur  = conn.cursor()

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        cur.execute("""
            INSERT INTO rag_chunks (source, chunk_index, content, embedding)
            VALUES (%s, %s, %s, %s)
        """, (doc_name, i, chunk, np.array(emb)))

    conn.commit()
    conn.close()
    return len(chunks)


def retrieve_context(query: str) -> list[dict]:
    """
    Embeds the query and retrieves top-K most similar chunks
    from rag_chunks using pgvector cosine distance (<=>).
    cosine_score = 1 - cosine_distance  (1.0 = identical)
    """
    query_embedding = embed_query(query)
    conn = get_pg_conn()
    cur  = conn.cursor()

    cur.execute("""
        SELECT
            content,
            source,
            chunk_index,
            ingested_at,
            1 - (embedding <=> %s) AS cosine_score
        FROM rag_chunks
        ORDER BY embedding <=> %s
        LIMIT %s
    """, (np.array(query_embedding), np.array(query_embedding), TOP_K))

    rows = cur.fetchall()
    conn.close()

    return [
        {
            "text":        r[0],
            "source":      r[1],
            "chunk_index": r[2],
            "ingested_at": str(r[3]) if r[3] else "",
            "score":       round(float(r[4]), 4),
        }
        for r in rows
    ]


print("✅ store_in_postgres() and retrieve_context() defined")
print("   Vector store: PostgreSQL 18 + pgvector (HNSW cosine)")

✅ store_in_postgres() and retrieve_context() defined
   Vector store: PostgreSQL 18 + pgvector (HNSW cosine)


In [6]:
# ── Cell 7: ▶️ RUN INGESTION ──────────────────────────────────────────────────
# Change PDF_PATH to your actual PDF file path.
# Re-run this cell to ingest additional PDFs — PostgreSQL appends, does not overwrite.

import os

PDF_PATH = "./pdfs/attention.pdf"   # ← CHANGE THIS

assert os.path.exists(PDF_PATH), f"❌ File not found: {PDF_PATH}"

doc_name = os.path.basename(PDF_PATH)
print(f"📄 Ingesting: {doc_name}")
print("-" * 50)

# Step 1: Extract
print("Step 1/3  Extracting text from PDF...")
raw_text, page_count = extract_text_from_pdf(PDF_PATH)
print(f"✅ {page_count} pages | {len(raw_text):,} characters")

# Step 2: Chunk
print("\nStep 2/3  Chunking text...")
chunks = chunk_text(raw_text)
print(f"✅ {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

# Step 3: Embed + Store
print(f"\nStep 3/3  Embedding via {GEMINI_EMBED_MODEL}...")
embeddings = embed_documents_batch(chunks)
stored     = store_in_postgres(chunks, embeddings, doc_name)

print("-" * 50)
print(f"✅ Done! {stored} chunks stored in PostgreSQL → rag_chunks table")

📄 Ingesting: attention.pdf
--------------------------------------------------
Step 1/3  Extracting text from PDF...
✅ 15 pages | 39,784 characters

Step 2/3  Chunking text...
✅ 59 chunks (size=800, overlap=120)

Step 3/3  Embedding via gemini-embedding-001...
  Embedding chunk 59/59...
  ✅ Embedded 59 chunks
--------------------------------------------------
✅ Done! 59 chunks stored in PostgreSQL → rag_chunks table


In [7]:
# ── Cell 8: gpt-oss-20b Answer Generation via HuggingFace ─────────────────────

from huggingface_hub import InferenceClient

hf_client = InferenceClient(api_key=HF_API_KEY)

SYSTEM_MSG = (
    "You are a precise, document-grounded assistant. "
    "Answer ONLY using the CONTEXT provided. "
    "If the context is insufficient, say: "
    "'The provided documents don't cover this sufficiently.' "
    "Never fabricate facts. Be concise but complete. "
    "Use bullet points for multi-part answers."
)


def build_messages(query: str, context_chunks: list[dict]) -> list[dict]:
    context_blocks = [
        f"[Source: {c['source']} | Chunk #{c['chunk_index']} | Relevance: {c['score']}]\n{c['text']}"
        for c in context_chunks
    ]
    context_str  = "\n\n---\n\n".join(context_blocks)
    user_content = (
        f"Use the following context to answer the question.\n\n"
        f"CONTEXT:\n{context_str}\n\n"
        f"QUESTION:\n{query}"
    )
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": user_content},
    ]


def generate_answer(messages: list[dict]) -> str:
    completion = hf_client.chat.completions.create(
        model=HF_LLM_MODEL,
        messages=messages,
        max_tokens=LLM_MAX_NEW_TOKENS,
        temperature=LLM_TEMPERATURE,
    )
    return completion.choices[0].message.content.strip()


print(f"✅ gpt-oss-20b generation ready")
print(f"   Model: {HF_LLM_MODEL}")

✅ gpt-oss-20b generation ready
   Model: openai/gpt-oss-20b:groq


In [8]:
# ── Cell 9: Full RAG Pipeline — ask() ─────────────────────────────────────────

def ask(query: str, show_chunks: bool = True):
    """
    Full RAG pipeline:
      query → embed → pgvector retrieval → gpt-oss-20b generation → answer
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {query}")
    print(f"{'='*60}")

    print("🔍 Retrieving relevant chunks from PostgreSQL...")
    chunks = retrieve_context(query)

    if not chunks:
        print("❌ No chunks found. Run Cell 7 (ingestion) first.")
        return

    if show_chunks:
        print(f"\n📚 Top {len(chunks)} retrieved chunks:")
        for i, c in enumerate(chunks, 1):
            filled = int(c["score"] * 30)
            bar    = "█" * filled + "░" * (30 - filled)
            print(f"  {i}. {c['source']} chunk #{c['chunk_index']}  [{bar}]  {c['score']}")

    print(f"\n💬 Generating answer via {HF_LLM_MODEL}...\n")
    messages = build_messages(query, chunks)

    try:
        answer = generate_answer(messages)
        print(f"{'─'*60}")
        print("✅ Answer:")
        print(f"{'─'*60}")
        print(answer)
        print(f"{'─'*60}")
        return answer
    except Exception as e:
        err = str(e)
        if "503" in err or "loading" in err.lower():
            print("⏳ Model cold-starting. Wait ~30s and re-run.")
        elif "429" in err or "rate" in err.lower():
            print("⚠️  Rate limit hit. Wait ~60s and retry.")
        else:
            print(f"❌ LLM error: {e}")


print("✅ ask() pipeline defined — ready to query")

✅ ask() pipeline defined — ready to query


In [9]:
# ── Cell 10: ▶️ ASK A SINGLE QUESTION ─────────────────────────────────────────
# Change the question below and re-run this cell anytime.

ask("What is the main topic of this document?")


❓ Question: What is the main topic of this document?
🔍 Retrieving relevant chunks from PostgreSQL...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #0  [███████████████████░░░░░░░░░░░]  0.6529
  2. attention.pdf chunk #56  [███████████████████░░░░░░░░░░░]  0.6498
  3. attention.pdf chunk #58  [███████████████████░░░░░░░░░░░]  0.6476
  4. attention.pdf chunk #57  [███████████████████░░░░░░░░░░░]  0.6474
  5. attention.pdf chunk #54  [███████████████████░░░░░░░░░░░]  0.6399

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
────────────────────────────────────────────────────────────
- The document is the research paper **“Attention Is All You Need,”** which introduces the Transformer architecture—a sequence‑transduction model that relies entirely on attention mechanisms instead of recurrent or convolutional networks.
────────────────────────────────────────────────────────────


'- The document is the research paper **“Attention Is All You Need,”** which introduces the Transformer architecture—a sequence‑transduction model that relies entirely on attention mechanisms instead of recurrent or convolutional networks.'

In [10]:
# ── Cell 11: ▶️ INTERACTIVE REPL LOOP ─────────────────────────────────────────

print("🚀 Local RAG — Interactive Mode")
print(f"   Embeddings : {GEMINI_EMBED_MODEL}")
print(f"   LLM        : {HF_LLM_MODEL}")
print(f"   Vector DB  : PostgreSQL 18 + pgvector")
print("   Type 'exit' or 'quit' to stop.\n")

while True:
    try:
        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in ("exit", "quit", "q"):
            print("👋 Goodbye!")
            break
        ask(query)
    except KeyboardInterrupt:
        print("\n👋 Interrupted. Goodbye!")
        break

🚀 Local RAG — Interactive Mode
   Embeddings : gemini-embedding-001
   LLM        : openai/gpt-oss-20b:groq
   Vector DB  : PostgreSQL 18 + pgvector
   Type 'exit' or 'quit' to stop.

👋 Goodbye!


In [11]:
# ── Cell 12: UTILITIES ────────────────────────────────────────────────────────

def check_db_stats():
    """Show row counts for all RAG tables."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    tables = ["rag_chunks", "rag_staleness", "rag_chunk_registry",
               "rag_sessions", "rag_turns"]
    print(f"\n📊 PostgreSQL RAG table stats")
    print(f"   {'Table':<25} {'Rows':>8}")
    print(f"   {'─'*35}")
    for t in tables:
        cur.execute(f"SELECT COUNT(*) FROM {t};")
        count = cur.fetchone()[0]
        print(f"   {t:<25} {count:>8}")
    conn.close()


def list_ingested_docs():
    """List all unique source documents in rag_chunks."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT source, COUNT(*) AS chunks, MIN(ingested_at) AS first_ingested
        FROM rag_chunks
        GROUP BY source
        ORDER BY first_ingested;
    """)
    rows = cur.fetchall()
    conn.close()
    print(f"\n📁 Ingested documents ({len(rows)}):")
    for r in rows:
        print(f"   - {r[0]}  ({r[1]} chunks, ingested {str(r[2])[:19]})")


def wipe_db():
    """Delete ALL rows from all RAG tables. Irreversible."""
    confirm = input("⚠️  This deletes ALL data from all RAG tables. Type 'yes' to confirm: ")
    if confirm.strip().lower() == "yes":
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("DELETE FROM rag_turns;")
        cur.execute("DELETE FROM rag_sessions;")
        cur.execute("DELETE FROM rag_chunk_registry;")
        cur.execute("DELETE FROM rag_staleness;")
        cur.execute("DELETE FROM rag_chunks;")
        conn.commit()
        conn.close()
        print("🗑️  All RAG tables wiped. Re-run ingestion to repopulate.")
    else:
        print("Cancelled.")


# ── Run whichever you need ────────────────────────────────────────────────────
check_db_stats()
# list_ingested_docs()
# wipe_db()


📊 PostgreSQL RAG table stats
   Table                         Rows
   ───────────────────────────────────
   rag_chunks                      59
   rag_staleness                    0
   rag_chunk_registry               0
   rag_sessions                     0
   rag_turns                        0


In [12]:
# ── Cell 13: PostgreSQL Direct Query Explorer ──────────────────────────────────

def peek(n: int = 5):
    """View the first N stored chunks."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT source, chunk_index, content, ingested_at
        FROM rag_chunks
        ORDER BY ingested_at, chunk_index
        LIMIT %s;
    """, (n,))
    rows = cur.fetchall()
    conn.close()
    print(f"\n📄 First {n} chunks:\n{'─'*60}")
    for r in rows:
        print(f"[{r[1]}] {r[0]}  |  {str(r[3])[:19]}")
        print(f"     {r[2][:200]}...\n")


def get_chunks_by_source(source_name: str):
    """Retrieve all chunks from a specific ingested PDF."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT chunk_index, content
        FROM rag_chunks
        WHERE source = %s
        ORDER BY chunk_index;
    """, (source_name,))
    rows = cur.fetchall()
    conn.close()
    print(f"\n📁 Chunks from '{source_name}': {len(rows)} found\n{'─'*60}")
    for r in rows:
        print(f"  Chunk #{r[0]:>3} | {r[1][:150]}...")


def get_chunk(source_name: str, chunk_index: int):
    """Retrieve and print a specific chunk by source + index."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT content, ingested_at
        FROM rag_chunks
        WHERE source = %s AND chunk_index = %s;
    """, (source_name, chunk_index))
    row = cur.fetchone()
    conn.close()
    if not row:
        print(f"❌ No chunk found: {source_name} #{chunk_index}")
        return
    print(f"\n🔎 {source_name} | Chunk #{chunk_index} | {str(row[1])[:19]}\n{'─'*60}")
    print(row[0])


def semantic_search(query: str, top_k: int = TOP_K):
    """Embed query and find similar chunks. Shows raw scores — no LLM."""
    chunks = retrieve_context(query)
    print(f"\n🔍 Semantic search: '{query}'\n{'─'*60}")
    for c in chunks:
        filled = int(c["score"] * 30)
        bar    = "█" * filled + "░" * (30 - filled)
        print(f"  [{bar}] {c['score']} | {c['source']} chunk #{c['chunk_index']}")
        print(f"  {c['text'][:250]}...\n")


def keyword_search(keyword: str):
    """PostgreSQL full-text ILIKE search across all chunks."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT source, chunk_index, content
        FROM rag_chunks
        WHERE content ILIKE %s
        ORDER BY source, chunk_index;
    """, (f"%{keyword}%",))
    rows = cur.fetchall()
    conn.close()
    print(f"\n🔎 Keyword: '{keyword}' — {len(rows)} chunk(s) matched\n{'─'*60}")
    for r in rows:
        idx     = r[2].lower().find(keyword.lower())
        snippet = r[2][max(0, idx-80):idx+120]
        print(f"  {r[0]} chunk #{r[1]}:")
        print(f"  ...{snippet}...\n")


def export_to_json(output_path: str = "./pg_export.json"):
    """Export all chunks to a JSON file."""
    import json
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("SELECT source, chunk_index, content, ingested_at FROM rag_chunks ORDER BY source, chunk_index;")
    rows = cur.fetchall()
    conn.close()
    records = [
        {"source": r[0], "chunk_index": r[1], "text": r[2], "ingested_at": str(r[3])}
        for r in rows
    ]
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print(f"✅ Exported {len(records)} chunks → {output_path}")


# ── Run whichever you need ────────────────────────────────────────────────────
peek(n=3)
# get_chunks_by_source("attention.pdf")
# get_chunk("attention.pdf", chunk_index=6)
# semantic_search("multi-head attention")
# keyword_search("BLEU")
# export_to_json()


📄 First 3 chunks:
────────────────────────────────────────────────────────────
[0] attention.pdf  |  2026-04-27 10:44:51
     [Page 1]
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All ...

[1] attention.pdf  |  2026-04-27 10:44:51
     ent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple netw...

[2] attention.pdf  |  2026-04-27 10:44:51
     -to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best ...



In [13]:
# ── Cell 13A: Conversation Memory Manager (PostgreSQL-backed) ──────────────────
# All session data lives in rag_sessions and rag_turns tables.
# No JSON files on disk — everything queryable via SQL.

from datetime import datetime

MAX_HISTORY_TURNS = 6


class ConversationMemory:
    """
    PostgreSQL-backed conversation memory.
    Each session is a row in rag_sessions.
    Each turn is a row in rag_turns.
    Auto-checkpoints after every answer — no data loss on kernel crash.
    """

    def __init__(self, session_id: str = None):
        self.session_id = session_id or datetime.now().strftime("%Y%m%d_%H%M%S")
        conn = get_pg_conn()
        cur  = conn.cursor()

        # Check if session already exists
        cur.execute("SELECT turn_count FROM rag_sessions WHERE session_id = %s;",
                    (self.session_id,))
        row = cur.fetchone()

        if row:
            # Resume existing session
            self._turn_count = row[0]
            conn.close()
            print(f"📂 Resumed session: {self.session_id}")
            print(f"   {self._turn_count} previous turns loaded")
        else:
            # Create new session
            cur.execute("""
                INSERT INTO rag_sessions (session_id, model, embed_model)
                VALUES (%s, %s, %s);
            """, (self.session_id, HF_LLM_MODEL, GEMINI_EMBED_MODEL))
            conn.commit()
            conn.close()
            self._turn_count = 0
            print(f"🆕 New session: {self.session_id}")

    @property
    def metadata(self):
        return {"turns": self._turn_count, "session_id": self.session_id}

    def add_turn(self, question: str, answer: str, chunks: list[dict]):
        """Persist a Q&A turn to PostgreSQL immediately."""
        sources = [f"{c['source']} chunk #{c['chunk_index']} ({c['score']})"
                   for c in chunks]
        conn = get_pg_conn()
        cur  = conn.cursor()
        # Insert user turn
        cur.execute("""
            INSERT INTO rag_turns (session_id, role, content)
            VALUES (%s, 'user', %s);
        """, (self.session_id, question))
        # Insert assistant turn with sources
        cur.execute("""
            INSERT INTO rag_turns (session_id, role, content, sources)
            VALUES (%s, 'assistant', %s, %s);
        """, (self.session_id, answer, sources))
        # Update session metadata
        self._turn_count += 1
        cur.execute("""
            UPDATE rag_sessions
            SET turn_count = %s, updated_at = NOW()
            WHERE session_id = %s;
        """, (self._turn_count, self.session_id))
        conn.commit()
        conn.close()

    def get_messages_with_history(
        self, query: str, context_chunks: list[dict], system_msg: str
    ) -> list[dict]:
        """Builds prompt with last N turns injected as history."""
        # Fetch last N turns from PostgreSQL
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            SELECT role, content
            FROM rag_turns
            WHERE session_id = %s
            ORDER BY created_at DESC
            LIMIT %s;
        """, (self.session_id, MAX_HISTORY_TURNS * 2))
        rows = cur.fetchall()
        conn.close()

        # Reverse to chronological order
        history = list(reversed(rows))

        messages = [{"role": "system", "content": system_msg}]
        for role, content in history:
            messages.append({"role": role, "content": content})

        context_str  = "\n\n---\n\n".join([
            f"[Source: {c['source']} | Chunk #{c['chunk_index']} | Relevance: {c['score']}]\n{c['text']}"
            for c in context_chunks
        ])
        user_content = (
            f"Use the following context to answer the question. "
            f"You may also use our previous conversation if relevant.\n\n"
            f"CONTEXT:\n{context_str}\n\n"
            f"QUESTION:\n{query}"
        )
        messages.append({"role": "user", "content": user_content})
        return messages

    def show_history(self):
        """Print conversation history from PostgreSQL."""
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            SELECT role, content, sources, created_at
            FROM rag_turns
            WHERE session_id = %s
            ORDER BY created_at;
        """, (self.session_id,))
        rows = cur.fetchall()
        conn.close()
        if not rows:
            print("📭 No conversation history yet.")
            return
        print(f"\n📜 Session: {self.session_id} | {self._turn_count} turns\n{'─'*60}")
        for role, content, sources, ts in rows:
            icon = "🧑 You" if role == "user" else "🤖 AI"
            print(f"{icon} [{str(ts)[:19]}]:")
            print(f"  {content[:300]}{'...' if len(content) > 300 else ''}")
            if role == "assistant" and sources:
                print(f"  📎 Sources: {', '.join(sources[:3])}")
            print()

    def clear(self):
        """Delete all turns for this session."""
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("DELETE FROM rag_turns WHERE session_id = %s;", (self.session_id,))
        cur.execute("UPDATE rag_sessions SET turn_count = 0 WHERE session_id = %s;",
                    (self.session_id,))
        conn.commit()
        conn.close()
        self._turn_count = 0
        print(f"🗑️  History cleared for session: {self.session_id}")

    def summary(self):
        print(f"\n📊 Session summary")
        print(f"   ID        : {self.session_id}")
        print(f"   Turns     : {self._turn_count}")
        print(f"   Model     : {HF_LLM_MODEL}")
        print(f"   Storage   : PostgreSQL rag_sessions / rag_turns")
        print(f"   Max history sent to LLM: {MAX_HISTORY_TURNS} turns")


def list_sessions():
    """List all saved sessions from PostgreSQL."""
    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT session_id, turn_count, created_at, updated_at
        FROM rag_sessions
        ORDER BY created_at DESC;
    """)
    rows = cur.fetchall()
    conn.close()
    if not rows:
        print("📭 No saved sessions found.")
        return
    print(f"\n💾 Saved sessions ({len(rows)}):\n{'─'*60}")
    for r in rows:
        print(f"  📁 {r[0]}")
        print(f"     Turns   : {r[1]}")
        print(f"     Created : {str(r[2])[:19]}")
        print(f"     Updated : {str(r[3])[:19]}")
        print()


print("✅ ConversationMemory (PostgreSQL-backed) defined")
print(f"   Storage: rag_sessions + rag_turns tables")
print(f"   Max history per prompt: {MAX_HISTORY_TURNS} turns")

✅ ConversationMemory (PostgreSQL-backed) defined
   Storage: rag_sessions + rag_turns tables
   Max history per prompt: 6 turns


In [14]:
# ── Cell 13B: Memory-aware ask_with_memory() ───────────────────────────────────

SYSTEM_MSG_MEMORY = (
    "You are a precise, document-grounded assistant with memory of our conversation. "
    "Answer using ONLY the CONTEXT provided and prior conversation turns if relevant. "
    "If the context is insufficient, say: "
    "'The provided documents don't cover this sufficiently.' "
    "Never fabricate facts. Be concise but complete. "
    "Use bullet points for multi-part answers."
)


def ask_with_memory(query: str, memory: ConversationMemory, show_chunks: bool = True):
    print(f"\n{'='*60}")
    print(f"❓ Question: {query}")
    print(f"   Session : {memory.session_id} | Turn #{memory.metadata['turns'] + 1}")
    print(f"{'='*60}")

    print("🔍 Retrieving relevant chunks from PostgreSQL...")
    chunks = retrieve_context(query)

    if not chunks:
        print("❌ No chunks found. Run ingestion first.")
        return

    if show_chunks:
        print(f"\n📚 Top {len(chunks)} retrieved chunks:")
        for i, c in enumerate(chunks, 1):
            filled = int(c["score"] * 30)
            bar    = "█" * filled + "░" * (30 - filled)
            print(f"  {i}. {c['source']} chunk #{c['chunk_index']}  [{bar}]  {c['score']}")

    print(f"\n💬 Generating answer via {HF_LLM_MODEL}...\n")
    messages = memory.get_messages_with_history(query, chunks, SYSTEM_MSG_MEMORY)

    try:
        answer = generate_answer(messages)
        print(f"{'─'*60}")
        print("✅ Answer:")
        print(f"{'─'*60}")
        print(answer)
        print(f"{'─'*60}")

        memory.add_turn(question=query, answer=answer, chunks=chunks)
        print(f"💾 Checkpointed to PostgreSQL (turn #{memory.metadata['turns']})")
        return answer
    except Exception as e:
        print(f"❌ LLM error: {e}")


print("✅ ask_with_memory() defined")

✅ ask_with_memory() defined


In [15]:
# ── Cell 13C: ▶️ Interactive Memory REPL ──────────────────────────────────────

memory = ConversationMemory()   # new session
# memory = ConversationMemory("paste_session_id_here")  # resume existing

print()
memory.summary()

print("\n🚀 Memory-aware RAG — Interactive Mode")
print("   Commands: 'history' | 'summary' | 'clear' | 'sessions' | 'exit'\n")

while True:
    try:
        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in ("exit", "quit", "q"):
            print("👋 Goodbye! Session saved to PostgreSQL.")
            break
        elif query.lower() == "history":
            memory.show_history()
        elif query.lower() == "summary":
            memory.summary()
        elif query.lower() == "clear":
            memory.clear()
        elif query.lower() == "sessions":
            list_sessions()
        else:
            ask_with_memory(query, memory)
    except KeyboardInterrupt:
        print("\n👋 Interrupted. Session saved to PostgreSQL.")
        break

🆕 New session: 20260427_104637


📊 Session summary
   ID        : 20260427_104637
   Turns     : 0
   Model     : openai/gpt-oss-20b:groq
   Storage   : PostgreSQL rag_sessions / rag_turns
   Max history sent to LLM: 6 turns

🚀 Memory-aware RAG — Interactive Mode
   Commands: 'history' | 'summary' | 'clear' | 'sessions' | 'exit'

👋 Goodbye! Session saved to PostgreSQL.


In [16]:
# ── Cell 13D: ▶️ Initialize Memory Session ────────────────────────────────────
# Run once per session before asking questions.

memory = ConversationMemory()                              # new session
# memory = ConversationMemory("paste_session_id_here")   # resume existing

memory.summary()

🆕 New session: 20260427_104707

📊 Session summary
   ID        : 20260427_104707
   Turns     : 0
   Model     : openai/gpt-oss-20b:groq
   Storage   : PostgreSQL rag_sessions / rag_turns
   Max history sent to LLM: 6 turns


In [17]:
# ── Cell 13E: ▶️ Ask a Single Question With Memory ────────────────────────────

ask_with_memory("What is the main topic of this document?", memory)


❓ Question: What is the main topic of this document?
   Session : 20260427_104707 | Turn #1
🔍 Retrieving relevant chunks from PostgreSQL...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #0  [███████████████████░░░░░░░░░░░]  0.6529
  2. attention.pdf chunk #56  [███████████████████░░░░░░░░░░░]  0.6498
  3. attention.pdf chunk #58  [███████████████████░░░░░░░░░░░]  0.6476
  4. attention.pdf chunk #57  [███████████████████░░░░░░░░░░░]  0.6474
  5. attention.pdf chunk #54  [███████████████████░░░░░░░░░░░]  0.6399

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
────────────────────────────────────────────────────────────
- The document is the research paper **“Attention Is All You Need.”**  
- It introduces the **Transformer architecture**, a new sequence‑to‑sequence model that relies entirely on self‑attention mechanisms, replacing traditional recurrent or convolutional networks.
────────────────────────────

'- The document is the research paper **“Attention Is All You Need.”**  \n- It introduces the **Transformer architecture**, a new sequence‑to‑sequence model that relies entirely on self‑attention mechanisms, replacing traditional recurrent or convolutional networks.'

In [18]:
# ── Cell 13F: ▶️ Ask Multiple Questions With Memory ───────────────────────────

questions = [
    "What architecture does this paper propose?",
    "How many attention heads does it use?",
    "What optimizer was used for training?",
    "How does that compare to what you said about the architecture?",  # follow-up
]

for q in questions:
    ask_with_memory(q, memory)


❓ Question: What architecture does this paper propose?
   Session : 20260427_104707 | Turn #2
🔍 Retrieving relevant chunks from PostgreSQL...

📚 Top 5 retrieved chunks:
  1. attention.pdf chunk #6  [█████████████████████░░░░░░░░░]  0.7242
  2. attention.pdf chunk #9  [█████████████████████░░░░░░░░░]  0.7182
  3. attention.pdf chunk #10  [█████████████████████░░░░░░░░░]  0.7162
  4. attention.pdf chunk #1  [█████████████████████░░░░░░░░░]  0.7152
  5. attention.pdf chunk #11  [█████████████████████░░░░░░░░░]  0.7079

💬 Generating answer via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
✅ Answer:
────────────────────────────────────────────────────────────
- The paper proposes the **Transformer architecture**:  
  * An encoder‑decoder model that uses only self‑attention mechanisms (multi‑head attention) and position‑wise feed‑forward layers.  
  * It eliminates recurrence and convolution, enabling greater parallelization and faster training whi

In [19]:
# ── Cell 14A: Staleness Tracking (PostgreSQL-backed) ──────────────────────────
# Tracks per-document: file hash, ingest timestamp, version.
# All state in rag_staleness and rag_chunk_registry tables — no JSON files.

import hashlib
import uuid as _uuid


# ── Hash utilities ────────────────────────────────────────────────────────────
def compute_file_hash(pdf_path: str) -> str:
    sha = hashlib.sha256()
    with open(pdf_path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            sha.update(block)
    return sha.hexdigest()


def compute_chunk_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


# ── Staleness Registry (PostgreSQL) ──────────────────────────────────────────
class StalenessRegistry:
    def register(self, doc_name: str, file_hash: str, chunk_count: int):
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            INSERT INTO rag_staleness (doc_name, file_hash, chunk_count, version)
            VALUES (%s, %s, %s, 1)
            ON CONFLICT (doc_name) DO UPDATE SET
                file_hash   = EXCLUDED.file_hash,
                chunk_count = EXCLUDED.chunk_count,
                ingested_at = NOW(),
                version     = rag_staleness.version + 1;
        """, (doc_name, file_hash, chunk_count))
        conn.commit()
        conn.close()

    def check(self, pdf_path: str) -> dict:
        doc_name     = os.path.basename(pdf_path)
        current_hash = compute_file_hash(pdf_path)
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            SELECT file_hash, ingested_at, version
            FROM rag_staleness WHERE doc_name = %s;
        """, (doc_name,))
        row = cur.fetchone()
        conn.close()

        if not row:
            return {"status": "new", "doc_name": doc_name, "file_hash": current_hash}
        if row[0] == current_hash:
            return {"status": "unchanged", "doc_name": doc_name,
                    "file_hash": current_hash,
                    "ingested_at": str(row[1]), "version": row[2]}
        return {"status": "stale", "doc_name": doc_name,
                "file_hash": current_hash, "old_hash": row[0],
                "ingested_at": str(row[1]), "version": row[2]}

    def show(self):
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("SELECT doc_name, version, ingested_at, chunk_count, file_hash FROM rag_staleness;")
        rows = cur.fetchall()
        conn.close()
        if not rows:
            print("📭 Staleness registry is empty.")
            return
        print(f"\n📋 Staleness Registry ({len(rows)} docs)\n{'─'*60}")
        for r in rows:
            print(f"  ✅ {r[0]}")
            print(f"     Version  : v{r[1]}")
            print(f"     Ingested : {str(r[2])[:19]}")
            print(f"     Chunks   : {r[3]}")
            print(f"     Hash     : {r[4][:24]}...")
            print()


# ── Chunk Registry (PostgreSQL) ───────────────────────────────────────────────
class ChunkRegistry:
    def get_doc(self, doc_name: str) -> dict:
        """Returns {chunk_hash: chunk_id} for a document."""
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            SELECT chunk_hash, chunk_id FROM rag_chunk_registry
            WHERE doc_name = %s;
        """, (doc_name,))
        rows = cur.fetchall()
        conn.close()
        return {r[0]: r[1] for r in rows}

    def update(self, doc_name: str, registry: dict):
        """Replace the registry for a document with a new {chunk_hash: chunk_id} dict."""
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("DELETE FROM rag_chunk_registry WHERE doc_name = %s;", (doc_name,))
        for chunk_hash, chunk_id in registry.items():
            cur.execute("""
                INSERT INTO rag_chunk_registry (doc_name, chunk_hash, chunk_id)
                VALUES (%s, %s, %s);
            """, (doc_name, chunk_hash, chunk_id))
        conn.commit()
        conn.close()

    def show(self):
        conn = get_pg_conn()
        cur  = conn.cursor()
        cur.execute("""
            SELECT doc_name, COUNT(*) FROM rag_chunk_registry GROUP BY doc_name;
        """)
        rows = cur.fetchall()
        conn.close()
        if not rows:
            print("📭 Chunk registry is empty.")
            return
        print(f"\n🗂️  Chunk Registry ({len(rows)} docs):")
        for r in rows:
            print(f"   {r[0]}: {r[1]} chunks tracked")


# ── Enhanced store with file_hash + ingested_at ───────────────────────────────
def store_in_postgres_v2(
    chunks:      list[str],
    embeddings:  list[list[float]],
    doc_name:    str,
    file_hash:   str,
    ingested_at: str,
) -> dict:
    """
    Inserts chunks with full metadata into rag_chunks.
    Returns {chunk_hash: chunk_id} for CDC tracking.
    """
    conn = get_pg_conn()
    cur  = conn.cursor()
    hash_to_id = {}

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        chunk_id = str(_uuid.uuid4())
        cur.execute("""
            INSERT INTO rag_chunks
                (id, source, chunk_index, content, file_hash, ingested_at, embedding)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (chunk_id, doc_name, i, chunk, file_hash, ingested_at, np.array(emb)))
        hash_to_id[compute_chunk_hash(chunk)] = chunk_id

    conn.commit()
    conn.close()
    return hash_to_id


# ── Initialise ────────────────────────────────────────────────────────────────
staleness_reg = StalenessRegistry()
chunk_reg     = ChunkRegistry()

print("✅ Cell 14A: Staleness + Chunk registries ready (PostgreSQL)")
staleness_reg.show()
chunk_reg.show()

✅ Cell 14A: Staleness + Chunk registries ready (PostgreSQL)
📭 Staleness registry is empty.
📭 Chunk registry is empty.


In [20]:
# ── Cell 14B: CDC Engine + Smart Ingest (PostgreSQL) ──────────────────────────

from datetime import datetime


def diff_chunks(old_registry: dict, new_chunks: list[str]) -> dict:
    new_hash_map = {compute_chunk_hash(c): c for c in new_chunks}
    old_hashes   = set(old_registry.keys())
    new_hashes   = set(new_hash_map.keys())
    return {
        "added":     {h: new_hash_map[h] for h in (new_hashes - old_hashes)},
        "removed":   {h: old_registry[h] for h in (old_hashes - new_hashes)},
        "unchanged": {h: old_registry[h] for h in (old_hashes & new_hashes)},
    }


def cdc_update(pdf_path: str) -> dict:
    """
    Surgical CDC update for a changed PDF:
      1. Extract + chunk new version
      2. Diff chunk hashes vs stored registry
      3. DELETE removed chunks from rag_chunks by UUID
      4. Embed + INSERT only added chunks
      5. Update rag_staleness + rag_chunk_registry
    """
    doc_name    = os.path.basename(pdf_path)
    file_hash   = compute_file_hash(pdf_path)
    ingested_at = datetime.now().isoformat()

    print("\n📄 Extracting new version...")
    raw_text, page_count = extract_text_from_pdf(pdf_path)
    new_chunks = chunk_text(raw_text)
    print(f"   {page_count} pages → {len(new_chunks)} chunks")

    old_registry = chunk_reg.get_doc(doc_name)
    diff         = diff_chunks(old_registry, new_chunks)

    n_added     = len(diff["added"])
    n_removed   = len(diff["removed"])
    n_unchanged = len(diff["unchanged"])
    total       = n_unchanged + n_added

    print(f"\n📊 CDC diff result:")
    print(f"   ✅ Unchanged : {n_unchanged:>4}  (kept — zero re-embedding cost)")
    print(f"   ➕ Added     : {n_added:>4}  (will be embedded + inserted)")
    print(f"   ➖ Removed   : {n_removed:>4}  (will be deleted from rag_chunks)")
    if total > 0:
        print(f"   💰 API calls saved: {n_unchanged}/{total} ({int(100*n_unchanged/total)}% reuse)")

    conn = get_pg_conn()
    cur  = conn.cursor()

    # Delete removed chunks by UUID
    if diff["removed"]:
        ids_to_delete = list(diff["removed"].values())
        cur.execute("""
            DELETE FROM rag_chunks WHERE id = ANY(%s::uuid[]);
        """, (ids_to_delete,))
        print(f"\n🗑️  Deleted {len(ids_to_delete)} stale chunks from rag_chunks")

    conn.commit()
    conn.close()

    new_doc_registry = dict(diff["unchanged"])

    # Embed + insert added chunks
    if diff["added"]:
        added_texts = list(diff["added"].values())
        print(f"\n🧠 Embedding {len(added_texts)} new/changed chunks...")
        new_embeddings = embed_documents_batch(added_texts)

        conn = get_pg_conn()
        cur  = conn.cursor()
        # Get next available chunk_index for this doc
        cur.execute("""
            SELECT COALESCE(MAX(chunk_index), -1) FROM rag_chunks WHERE source = %s;
        """, (doc_name,))
        next_idx = cur.fetchone()[0] + 1

        for i, (h, (text, emb)) in enumerate(
            zip(diff["added"].keys(),
                zip(added_texts, new_embeddings))
        ):
            cid = str(_uuid.uuid4())
            cur.execute("""
                INSERT INTO rag_chunks
                    (id, source, chunk_index, content, file_hash, ingested_at, embedding)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
            """, (cid, doc_name, next_idx+i, text, file_hash, ingested_at, np.array(emb)))
            new_doc_registry[h] = cid

        conn.commit()
        conn.close()
        print(f"✅ Inserted {len(added_texts)} new chunks")

    chunk_reg.update(doc_name, new_doc_registry)
    staleness_reg.register(doc_name, file_hash, len(new_doc_registry))
    print(f"\n💾 CDC complete — rag_staleness + rag_chunk_registry updated")
    return {"added": n_added, "removed": n_removed, "unchanged": n_unchanged}


def ingest_pdf_v2(pdf_path: str, force: bool = False):
    """Smart ingest: new → full ingest | stale → CDC | unchanged → skip."""
    doc_name = os.path.basename(pdf_path)
    print(f"\n{'='*60}")
    print(f"📥 Smart Ingest: {doc_name}")
    print(f"{'='*60}")

    status = staleness_reg.check(pdf_path)
    print(f"📌 Status: {status['status'].upper()}")

    if status["status"] == "unchanged" and not force:
        print(f"✅ Already up to date (v{status['version']})")
        print("   Use force=True to re-ingest anyway.")
        return

    if status["status"] == "stale":
        print(f"🔄 Changes detected (was v{status['version']}) — running CDC...")
        return cdc_update(pdf_path)

    # New or forced — full ingest
    ingested_at = datetime.now().isoformat()
    file_hash   = status["file_hash"]

    print("📄 Step 1/3  Extracting text...")
    raw_text, page_count = extract_text_from_pdf(pdf_path)
    print(f"   ✅ {page_count} pages | {len(raw_text):,} chars")

    print("✂️  Step 2/3  Chunking...")
    chunks = chunk_text(raw_text)
    print(f"   ✅ {len(chunks)} chunks")

    print(f"🧠 Step 3/3  Embedding via {GEMINI_EMBED_MODEL}...")
    embeddings = embed_documents_batch(chunks)

    chunk_hash_map = store_in_postgres_v2(
        chunks, embeddings, doc_name, file_hash, ingested_at
    )
    chunk_reg.update(doc_name, chunk_hash_map)
    staleness_reg.register(doc_name, file_hash, len(chunks))

    print(f"\n✅ Full ingest complete! {len(chunks)} chunks in rag_chunks")
    print(f"   Hash: {file_hash[:24]}... | Ingested: {ingested_at[:19]}")


print("✅ Cell 14B: CDC Engine + Smart Ingest ready")

✅ Cell 14B: CDC Engine + Smart Ingest ready


In [21]:
# ── Cell 14C: Recency-Weighted Retrieval (PostgreSQL) ─────────────────────────
# Blends cosine similarity with exponential decay directly in Python.
# PostgreSQL fetches TOP_K*3 candidates; recency re-ranking done in Python.

import math
from datetime import datetime, timezone

RECENCY_ALPHA     = 0.85
RECENCY_HALF_LIFE = 30


def recency_decay(ingested_at_str: str,
                  half_life_days: float = RECENCY_HALF_LIFE) -> float:
    try:
        ingested  = datetime.fromisoformat(str(ingested_at_str).replace("Z", "+00:00"))
        if ingested.tzinfo is None:
            ingested = ingested.replace(tzinfo=timezone.utc)
        days_gone = (datetime.now(timezone.utc) - ingested).total_seconds() / 86400
        lam       = math.log(2) / half_life_days
        return round(math.exp(-lam * days_gone), 4)
    except Exception:
        return 0.5


def retrieve_context_weighted(
    query:       str,
    alpha:       float = RECENCY_ALPHA,
    show_scores: bool  = False,
) -> list[dict]:
    """
    Fetches TOP_K*3 candidates by cosine similarity from PostgreSQL,
    then re-ranks by: alpha * cosine + (1-alpha) * recency_decay
    Returns top TOP_K after blended re-ranking.
    """
    query_embedding = embed_query(query)
    n_candidates    = TOP_K * 3

    conn = get_pg_conn()
    cur  = conn.cursor()
    cur.execute("""
        SELECT
            content, source, chunk_index, ingested_at,
            1 - (embedding <=> %s) AS cosine_score
        FROM rag_chunks
        ORDER BY embedding <=> %s
        LIMIT %s;
    """, (np.array(query_embedding), np.array(query_embedding), n_candidates))
    rows = cur.fetchall()
    conn.close()

    chunks = []
    for r in rows:
        cosine_score  = round(float(r[4]), 4)
        ingested_at   = str(r[3]) if r[3] else datetime.now(timezone.utc).isoformat()
        recency_score = recency_decay(ingested_at)
        blended       = round(alpha * cosine_score + (1 - alpha) * recency_score, 4)
        chunks.append({
            "text":          r[0],
            "source":        r[1],
            "chunk_index":   r[2],
            "ingested_at":   str(r[3])[:19] if r[3] else "",
            "cosine_score":  cosine_score,
            "recency_score": recency_score,
            "score":         blended,
        })

    chunks.sort(key=lambda x: x["score"], reverse=True)
    top_chunks = chunks[:TOP_K]

    if show_scores:
        print(f"\n📊 Score breakdown (α={alpha} cosine | {round(1-alpha,2)} recency | "
              f"half-life={RECENCY_HALF_LIFE}d):")
        print(f"   {'#':>3}  {'chunk':>8}  {'cosine':>8}  {'recency':>8}  {'blended':>8}  ingested")
        print(f"   {'─'*65}")
        for i, c in enumerate(top_chunks, 1):
            print(f"   {i:>3}  #{c['chunk_index']:>6}  "
                  f"{c['cosine_score']:>8}  "
                  f"{c['recency_score']:>8}  "
                  f"{c['score']:>8}  "
                  f"{c['ingested_at'][5:16]}")

    return top_chunks


def ask_weighted(
    query:       str,
    memory:      object = None,
    alpha:       float  = RECENCY_ALPHA,
    show_chunks: bool   = True,
    show_scores: bool   = False,
):
    print(f"\n{'='*60}")
    print(f"❓ {query}")
    if memory:
        print(f"   Session : {memory.session_id} | Turn #{memory.metadata['turns'] + 1}")
    print(f"   Retrieval: α={alpha} cosine | {round(1-alpha,2)} recency")
    print(f"{'='*60}")

    print("🔍 Retrieving with recency weighting from PostgreSQL...")
    chunks = retrieve_context_weighted(query, alpha=alpha, show_scores=show_scores)

    if not chunks:
        print("❌ No chunks found. Run ingest_pdf_v2() first.")
        return

    if show_chunks:
        print(f"\n📚 Top {len(chunks)} chunks:")
        for i, c in enumerate(chunks, 1):
            filled = int(c["score"] * 30)
            bar    = "█" * filled + "░" * (30 - filled)
            print(
                f"  {i}. {c['source']} #{c['chunk_index']:>3} "
                f"[{bar}] {c['score']}"
                f"  (cos={c['cosine_score']}, rec={c['recency_score']},"
                f" 📅{c['ingested_at'][5:16]})"
            )

    if memory:
        messages = memory.get_messages_with_history(query, chunks, SYSTEM_MSG_MEMORY)
    else:
        messages = build_messages(query, chunks)

    print(f"\n💬 Generating via {HF_LLM_MODEL}...\n")
    try:
        answer = generate_answer(messages)
        print(f"{'─'*60}")
        print("✅ Answer:")
        print(f"{'─'*60}")
        print(answer)
        print(f"{'─'*60}")

        if memory:
            memory.add_turn(query, answer, chunks)
            print(f"💾 Checkpointed to PostgreSQL (turn #{memory.metadata['turns']})")

        return answer
    except Exception as e:
        print(f"❌ LLM error: {e}")


print("✅ Cell 14C: Recency-Weighted Retrieval ready")
print(f"   Alpha  : {RECENCY_ALPHA} cosine | {round(1-RECENCY_ALPHA,2)} recency")
print(f"   Half-life: {RECENCY_HALF_LIFE} days")

✅ Cell 14C: Recency-Weighted Retrieval ready
   Alpha  : 0.85 cosine | 0.15 recency
   Half-life: 30 days


In [22]:
# ── Cell 14D: ▶️ Test PDF Generator ──────────────────────────────────────────
# Creates 3 versions of a RAG Guide PDF with controlled content changes.
# v1→v2: Vector DB section changed, LLM section added
# v2→v3: LLM section changed, Evaluation section added
# Introduction + Embedding Models: identical across all 3 — CDC should never re-embed these.

from fpdf import FPDF
import os, shutil

os.makedirs("./pdfs/versions", exist_ok=True)


def make_pdf(path: str, title: str, sections: dict):
    def sanitize(text: str) -> str:
        replacements = {
            "\u2014": "-", "\u2013": "-", "\u2018": "'", "\u2019": "'",
            "\u201c": '"', "\u201d": '"', "\u2026": "...", "\u00a0": " ",
        }
        for char, rep in replacements.items():
            text = text.replace(char, rep)
        return text.encode("latin-1", errors="ignore").decode("latin-1")

    pdf = FPDF()
    pdf.set_margins(20, 20, 20)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.cell(0, 14, sanitize(title), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)
    for heading, body in sections.items():
        pdf.set_font("Helvetica", "B", 13)
        pdf.cell(0, 10, sanitize(heading), new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", size=11)
        pdf.multi_cell(0, 7, sanitize(body))
        pdf.ln(5)
    pdf.output(path)
    print(f"✅ Created: {path}")


INTRO = (
    "Retrieval-Augmented Generation (RAG) combines retrieval with an LLM. "
    "Instead of relying on parametric knowledge alone, RAG fetches relevant "
    "documents and injects them into the model context. This grounds responses "
    "in verifiable source material and reduces hallucinations significantly."
)
EMBEDDING_MODELS = (
    "Embedding models map text to dense vectors capturing semantic meaning. "
    "Key models: Google gemini-embedding-001 (3072-dim, MTEB leaderboard #1), "
    "OpenAI text-embedding-3-large, and open models like BAAI/bge-large-en-v1.5. "
    "Asymmetric retrieval (RETRIEVAL_DOCUMENT vs RETRIEVAL_QUERY) is critical for RAG quality."
)
VECTOR_DB_v1 = (
    "Vector databases store high-dimensional embeddings and support ANN search. "
    "Popular options: ChromaDB (local, no server), FAISS (in-memory, Meta), "
    "Pinecone (managed cloud). HNSW provides sub-linear query time with high recall."
)
VECTOR_DB_v2 = (
    "Vector databases store high-dimensional embeddings and support ANN search. "
    "Leading options in 2025: ChromaDB, FAISS, Pinecone, Weaviate, Qdrant, "
    "and PostgreSQL with pgvector. pgvector is now a top production choice "
    "as it combines full SQL capability with native HNSW vector search."
)
LLM_INTEGRATION_v2 = (
    "The generation phase uses an LLM grounded in retrieved context. "
    "Key considerations: context window size, instruction following quality, "
    "and temperature (use 0.1-0.2 for factual RAG). Model options include "
    "GPT-4o, Gemini Flash, Llama 3.1-70B, and gpt-oss-20b."
)
LLM_INTEGRATION_v3 = (
    "The generation phase uses an LLM grounded in retrieved context. "
    "As of Q2 2025, leading RAG models include GPT-4o, Gemini 2.0 Flash, "
    "Llama 3.1-70B, and gpt-oss-20b. Re-ranking cross-encoders are now "
    "standard in production stacks, significantly improving precision."
)
RAG_EVALUATION_v3 = (
    "Evaluating RAG requires measuring both retrieval and generation quality. "
    "Retrieval metrics: MRR, NDCG, Recall@K. Generation metrics: Faithfulness, "
    "Answer Relevance, Context Precision. RAGAS automates evaluation using "
    "an LLM as judge. Faithfulness is the most critical production metric."
)

ACTIVE_PATH = "./pdfs/versions/rag_guide.pdf"
V2_STAGED   = "./pdfs/versions/rag_guide_v2_staged.pdf"
V3_STAGED   = "./pdfs/versions/rag_guide_v3_staged.pdf"

make_pdf(ACTIVE_PATH, "RAG Systems Guide - v1", {
    "1. Introduction to RAG": INTRO,
    "2. Vector Databases":    VECTOR_DB_v1,
    "3. Embedding Models":    EMBEDDING_MODELS,
})
make_pdf(V2_STAGED, "RAG Systems Guide - v2", {
    "1. Introduction to RAG": INTRO,
    "2. Vector Databases":    VECTOR_DB_v2,
    "3. Embedding Models":    EMBEDDING_MODELS,
    "4. LLM Integration":     LLM_INTEGRATION_v2,
})
make_pdf(V3_STAGED, "RAG Systems Guide - v3", {
    "1. Introduction to RAG": INTRO,
    "2. Vector Databases":    VECTOR_DB_v2,
    "3. Embedding Models":    EMBEDDING_MODELS,
    "4. LLM Integration":     LLM_INTEGRATION_v3,
    "5. RAG Evaluation":      RAG_EVALUATION_v3,
})

print("\n✅ All 3 test PDFs created in ./pdfs/versions/")
print("   v1→v2 : Vector DB updated, LLM section added")
print("   v2→v3 : LLM updated, Evaluation added")
print("   Introduction + Embedding: NEVER re-embedded across all versions")

✅ Created: ./pdfs/versions/rag_guide.pdf
✅ Created: ./pdfs/versions/rag_guide_v2_staged.pdf
✅ Created: ./pdfs/versions/rag_guide_v3_staged.pdf

✅ All 3 test PDFs created in ./pdfs/versions/
   v1→v2 : Vector DB updated, LLM section added
   v2→v3 : LLM updated, Evaluation added
   Introduction + Embedding: NEVER re-embedded across all versions


In [23]:
# ── Cell 14E: ▶️ Full Test Suite ──────────────────────────────────────────────

import shutil, time

ACTIVE_PATH = "./pdfs/versions/rag_guide.pdf"
V2_STAGED   = "./pdfs/versions/rag_guide_v2_staged.pdf"
V3_STAGED   = "./pdfs/versions/rag_guide_v3_staged.pdf"


# TEST 1 — Full ingest of v1
print("\n" + "█"*60)
print("  TEST 1 — Full ingest of v1")
print("█"*60)
ingest_pdf_v2(ACTIVE_PATH)
staleness_reg.show()
chunk_reg.show()


# TEST 2 — Staleness check (unchanged file)
print("\n" + "█"*60)
print("  TEST 2 — Staleness check (unchanged file)")
print("█"*60)
ingest_pdf_v2(ACTIVE_PATH)   # should say UNCHANGED and skip


# TEST 3 — Queries against v1
print("\n" + "█"*60)
print("  TEST 3 — Queries against v1")
print("█"*60)
ask_weighted("What vector databases does this guide mention?", show_scores=True)
ask_weighted("What is the recommended temperature for RAG?")


# TEST 4 — Upgrade to v2 (CDC)
print("\n" + "█"*60)
print("  TEST 4 — Upgrade to v2 (CDC update)")
print("█"*60)
print("📋 Copying v2 over active file...")
shutil.copy(V2_STAGED, ACTIVE_PATH)
time.sleep(1)
ingest_pdf_v2(ACTIVE_PATH)
staleness_reg.show()


# TEST 5 — Same queries after v2
print("\n" + "█"*60)
print("  TEST 5 — Same queries after v2 CDC update")
print("█"*60)
ask_weighted("What vector databases does this guide mention?", show_scores=True)
ask_weighted("What does the guide say about LLM integration?")


# TEST 6 — Upgrade to v3
print("\n" + "█"*60)
print("  TEST 6 — Upgrade to v3 (second CDC cycle)")
print("█"*60)
print("📋 Copying v3 over active file...")
shutil.copy(V3_STAGED, ACTIVE_PATH)
time.sleep(1)
ingest_pdf_v2(ACTIVE_PATH)


# TEST 7 — Recency verification
print("\n" + "█"*60)
print("  TEST 7 — Recency weighting verification")
print("█"*60)
ask_weighted("How should RAG pipelines be evaluated?", show_scores=True)
ask_weighted("What is Retrieval-Augmented Generation?", show_scores=True)


# TEST 8 — Full stack: recency + memory
print("\n" + "█"*60)
print("  TEST 8 — Full stack: weighted retrieval + PostgreSQL memory")
print("█"*60)
test_memory = ConversationMemory()
ask_weighted("What is RAG and what databases does this guide recommend?",
             memory=test_memory)
ask_weighted("What metrics should I use to evaluate it?",
             memory=test_memory)
ask_weighted("How does that compare to what you said about the databases?",
             memory=test_memory)


# SUMMARY
print("\n" + "█"*60)
print("  SUMMARY")
print("█"*60)
staleness_reg.show()
print("\n📊 You can verify all data directly in pgAdmin:")
print("   SELECT * FROM rag_chunks ORDER BY ingested_at;")
print("   SELECT * FROM rag_staleness;")
print("   SELECT * FROM rag_sessions;")
print("   SELECT * FROM rag_turns ORDER BY created_at;")


████████████████████████████████████████████████████████████
  TEST 1 — Full ingest of v1
████████████████████████████████████████████████████████████

📥 Smart Ingest: rag_guide.pdf
📌 Status: NEW
📄 Step 1/3  Extracting text...
   ✅ 1 pages | 916 chars
✂️  Step 2/3  Chunking...
   ✅ 2 chunks
🧠 Step 3/3  Embedding via gemini-embedding-001...
  Embedding chunk 2/2...
  ✅ Embedded 2 chunks

✅ Full ingest complete! 2 chunks in rag_chunks
   Hash: d1eee975c53d66788e257930... | Ingested: 2026-04-27T10:47:51

📋 Staleness Registry (1 docs)
────────────────────────────────────────────────────────────
  ✅ rag_guide.pdf
     Version  : v1
     Ingested : 2026-04-27 10:47:53
     Chunks   : 2
     Hash     : d1eee975c53d66788e257930...


🗂️  Chunk Registry (1 docs):
   rag_guide.pdf: 2 chunks tracked

████████████████████████████████████████████████████████████
  TEST 2 — Staleness check (unchanged file)
████████████████████████████████████████████████████████████

📥 Smart Ingest: rag_guide.pdf
📌 